# Week 3 -- Build the Full ADK Agent

## Problem Statement

You have a working deterministic baseline (Week 2). Its limit: hand-written keyword
rules can't cover the full variety of patient language. A patient says *"my head is
splitting and I keep being sick"* -- the rules miss "headache" because the exact keyword
isn't there. An LLM can handle this; a keyword matcher can't.

This week you build **6 specialised agents**. **5 of them** are wired into a
`SequentialAgent` pipeline (symptom_parser -> ... -> response_formatter); the **6th**,
`safety_evaluator`, runs as a **post-hoc audit layer** *after* the pipeline returns
(in Week 4 you reimplement it as a deterministic Python function). Each agent does one
job and writes its output to `session.state` so the next stage can read it.

The fourth agent, `triage_decider`, is the **agentic core**: it is given **4 FunctionTools**
(the tools you built in Week 2) and calls them mid-reasoning (ReAct) instead of guessing.

## What You Are Building This Week

You fill in **6 agent instruction prompts** (inside this notebook) and **assemble the pipeline**:

| Agent | output_key | In pipeline? | Your task |
|---|---|---|---|
| `symptom_parser` | `symptoms` | yes (1) | Write the extraction instruction |
| `severity_scorer` | `severity_json` | yes (2) | Write the scoring rubric |
| `followup_asker` | `followup` | yes (3) | Write the clarifying-question instruction |
| `triage_decider` | `triage_decision` | yes (4) -- **has 4 tools** | Write the decision instruction; it must CALL the tools |
| `response_formatter` | `final_response` | yes (5) | Write the response format instruction |
| `safety_evaluator` | `safety_audit` | no -- **post-hoc audit** | Write the compliance check instruction |

Then you assemble the **5 pipeline agents** into a `SequentialAgent` and run it.

## Learning Objectives

By the end of this week you will be able to:

1. Write a **constrained LLM agent instruction** that forces structured JSON output
2. Explain what `output_key` does and why each stage must write to a unique key
3. Wrap a Python function as a **FunctionTool** and give it to an agent (the ReAct pattern)
4. Describe the difference between a **SequentialAgent** (one-after-another) and
   calling LLMs independently
5. Run a live ADK evaluation and compare results to the Week 2 baseline

## Terminal Objectives (your deliverables)

- [ ] All 6 agent instructions written and non-empty
- [ ] `triage_decider` wired with its 4 FunctionTools
- [ ] `SequentialAgent` pipeline (5 agents) assembled and running
- [ ] `my_run_triage()` implemented
- [ ] At least one test case run end-to-end with output printed
- [ ] Week 2 vs Week 3 comparison note written

> **Cost awareness**: Each live ADK call = 5 LLM calls (one per pipeline agent), plus
> tool calls from `triage_decider`. Run the policy baseline first; use the ADK pipeline
> only when verifying. `utils.py` provides a ready `run_triage_async` helper.


<!-- ASSESSMENT_GUIDE v1 -->
## Week 3 — Assessment & Submission Guide  ·  29 marks

**Learning objectives — by the end of this notebook you can:**
- Finalise and document the six-agent architecture (jobs, I/O keys, the pause).
- Build the follow-up loop and prove it closes (the answer changes the decision).
- Implement the escalation-only decider and the safe response formatter.
- Implement the deterministic safety judge and pass the harness tests.
- Produce traced WAIT / DOCTOR / ER and answer-changes-decision demos.

**Files to modify & submit:**
- `week3_starter.ipynb` — write the six agent instructions and assemble the pipeline.
- Optional: copy completed instructions and `build_agentic_sahayak_pipeline()` to `sahayak_starter.py` to run `demo_app.py` with your own agents.

**Files provided for reference (do not submit):**
- `sahayak_tools.py`
- `tests/test_sahayak_harness.py`

**Depends on:** Weeks 1-2 (ADK foundations, dataset, baseline, parser/severity agents).

**Stage → Task → Sub-task → Marks → Expected output → Evidence**

| Task | Marks | Sub-task | Marks | Expected output |
|---|---:|---|---:|---|
| **1.5 Design the Agent Architecture** | **4** | 1.5.1 | 4 | All six agents specified (job, input keys, output key), the pause, escalate-never-de-escalate |
| **3.1 Follow-up Loop, Closed and Measured** | **8** | 3.1.1 | 3 | Follow-up asked only for severity 2-3; policy compliance >=90% | 
|  |  | 3.1.2 | 5 | Loop closes: pause, accept answer, decision changes; loop_target_compliance >=80% | 
| **3.2 Triage Decider & Safe Formatter** | **7** | 3.2.1 | 4 | Escalate on red-flags, never de-escalate (de_escalation_count = 0) | 
|  |  | 3.2.2 | 3 | Action-first response, exact disclaimer, no diagnosis/prescription |
| **3.3 Safety Evaluator & Deterministic Judge** | **6** | 3.3.1 | 4 | Deterministic judge with all six compliance checks (PASS/FLAG) | 
|  |  | 3.3.2 | 2 | Harness tests pass |
| **3.4 End-to-End Demos** | **4** | 3.4.1 | 4 | Four traced runs: WAIT, DOCTOR, ER, and answer-changes-decision | 
| | | | **29** | **Week 3 total** | 

**What counts as a completed deliverable:**
- The notebook executes top-to-bottom in Colab (Gemini) or locally (Ollama) with no errors.
- Every claimed number is visible as a notebook cell output (no separate .json artifacts required).
- Every sub-task above has visible evidence in the listed location.
- Week 4 only: attach `final_report.pdf` or `final_report.docx` covering methodology, eval results, failure analysis, known limits, and dashboard screenshots.

> Full grading guidelines (Award 100% / 50% / 0% per sub-task) are in `docs/GRADING_RUBRIC.csv`; the workflow context is in `docs/SAHAYAK_CAPSTONE_WORKFLOW.md` (Part E).

> **Note on task order:** the table above lists sub-tasks by topic. In the notebook the **build** steps (3.1.1, 3.2.1, 3.2.2, 3.3.1) come first; the two **measure** steps that need the fully-wired pipeline — **3.1.2** (loop closure) and **3.3.2** (harness tests) — run after it, just before the demos. So the cell order is *build → assemble → measure → demo*, not strict numeric order.

> **Priya's situation**: She spends 30 seconds per patient just writing down symptoms
> before she can think about urgency. That's 4 minutes wasted per 8-patient morning session.
> The agent you build this week gives those 4 minutes back -- if it works correctly.
> Your job: implement all 6 stages so the pipeline can run end-to-end without crashing.


## Concept Coverage -- Week 3

**Prerequisites from Weeks 1-2**: all 11 W1 concepts, trace table, eval set

| # | Concept | Type | Taught in | Your task |
|---|---------|------|-----------|----------|
| 1 | Writing a constrained `LlmAgent` instruction | ADK | W1 (shown) | FILL IN × 6 |
| 2 | `output_key` contract per stage | ADK | W1 (taught) | FILL IN: right key |
| 3 | Rule-locked prompt (explicit rules in instruction) | Prompt engineering | W1: scorer rules | FILL IN: encode rules |
| 4 | Assembling `SequentialAgent` | ADK | W1 (taught) | FILL IN: wire 6 agents |
| 5 | `Runner` + `InMemorySessionService` setup | ADK | W1 (shown) | FILL IN: recreate |
| 6 | State inspection for debugging | ADK | W1 (shown) | FILL IN: print all keys |
| 7 | 20-case batch evaluation | Evaluation | W2 (run with policy) | Repeat with ADK |
| 8 | Comparing ADK vs baseline | Evaluation | W2 (baseline locked) | Record delta |

**New in Week 3 (not seen before)**:
- Writing your own instruction text (W1 showed existing instructions; now you write them)
- `asyncio` event-loop pattern for running a full pipeline (W1 showed 2-agent; now 6-agent)

> **Worked example below** shows a fully written `symptom_parser` instruction.
> Use it as a template for the remaining 5 agents.


In [1]:
# >>> output-hygiene (HF/torch import advisories are not errors) >>>
import os as _os, logging as _logging, warnings as _warnings
for _k, _v in {"HF_HUB_DISABLE_IMPLICIT_TOKEN": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
               "HF_HUB_DISABLE_TELEMETRY": "1", "HF_HUB_VERBOSITY": "error",
               "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",
               "TOKENIZERS_PARALLELISM": "false"}.items():
    _os.environ.setdefault(_k, _v)
_warnings.filterwarnings("ignore")
for _n in ("huggingface_hub", "huggingface_hub.utils._http", "transformers",
           "sentence_transformers", "datasets", "torch",
           "torch.distributed.elastic.multiprocessing.redirects", "torchao"):
    _logging.getLogger(_n).setLevel(_logging.ERROR)
# <<< output-hygiene <<<
# -- COLAB SETUP ---------------------------------------------------------
# !pip install -q 'google-adk>=2.0.0' google-genai datasets pandas matplotlib seaborn scikit-learn
import os
# from google.colab import userdata
# os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
# os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'FALSE'
print('Setup done. Model auto-selects in the next cell: Gemini (with key) or local Ollama hermes3:8b.')

Setup done. Model auto-selects in the next cell: Gemini (with key) or local Ollama hermes3:8b.


In [2]:
# -- A5 safe .env loader ------------------------------------------------------
from pathlib import Path
import os

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for env_path in env_candidates:
    if not env_path.exists():
        continue
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip().removeprefix('export ').strip()
        value = value.strip().strip('"').strip("'")
        if key == 'GOOGLE_API_KEY' and value and not os.environ.get('GOOGLE_API_KEY'):
            os.environ['GOOGLE_API_KEY'] = value
    break

print('GOOGLE_API_KEY detected' if os.environ.get('GOOGLE_API_KEY') else 'GOOGLE_API_KEY missing')


GOOGLE_API_KEY detected


In [3]:
# -- MODEL SETUP -- auto-selects Gemini or Ollama ----------------------------
import os, re
from google.adk.models.lite_llm import LiteLlm

# Try Gemini first; fall back to local Ollama hermes3:8b if no key / quota.
# hermes3:8b is the same model used by demo_app.py and eval_agent.py.
GEMINI_KEY = os.getenv('GOOGLE_API_KEY', '')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'FALSE'

def _try_gemini(key):
    if not key or key == 'dummy':
        return False
    try:
        from google import genai
        os.environ['GOOGLE_API_KEY'] = key
        client = genai.Client()
        client.models.generate_content(model='gemini-flash-latest', contents='ping')
        return True
    except Exception:
        return False

if _try_gemini(GEMINI_KEY):
    MODEL = 'gemini-flash-latest'
    print('[OK] Using Gemini Flash latest')
else:
    os.environ['GOOGLE_API_KEY'] = 'dummy'
    MODEL = LiteLlm(model='ollama_chat/hermes3:8b', api_base='http://localhost:11434')
    print('[OK] Gemini unavailable -- using local Ollama hermes3:8b')

# Strip markdown fences local models sometimes add to JSON output
def clean_state(state: dict) -> dict:
    return {k: re.sub(r"^```[a-z]*\n?|```$", "", str(v).strip(), flags=re.MULTILINE).strip()
            for k, v in state.items()}

print(f'Model: {MODEL}')

[OK] Using Gemini Flash latest
Model: gemini-flash-latest


In [4]:
# -- Imports for Week 3 --------------------------------------------------------
import sys, asyncio, json, pandas as pd
sys.path.insert(0, ".")
sys.path.insert(0, "learner")

# -- data_loader.py -- GIVEN ---------------------------------------------------
#   build_evaluation_dataset()  -> the fixed 50-case eval split (same as Week 2)

# -- sahayak_starter.py -- YOUR FILE -------------------------------------------
#   DISCLAIMER  -> the required safety disclaimer text (a constant, not a stub)
#                 Every agent response must end with this. It is a hard contract.
from sahayak_starter import (
    DISCLAIMER,
    SYMPTOM_PARSER_INSTRUCTION,
    SEVERITY_SCORER_INSTRUCTION,
    FOLLOWUP_ASKER_INSTRUCTION,
    TRIAGE_DECIDER_INSTRUCTION,
    RESPONSE_FORMATTER_INSTRUCTION,
    SAFETY_EVALUATOR_INSTRUCTION,
    parse_predicted_triage,
)


## Agent Architecture Design

Before running any LLM agents, sketch the full six-agent pipeline you will build in
Week 3. Write your design in the cell below — it becomes the first section of your
architecture diagram in the final report.

Specify for each agent: **name · job · input key(s) · output key**.
Also state: (1) where the pipeline **pauses** for a follow-up (Phase A), and
(2) the **escalate-never-de-escalate rule** the decider must enforce.

<!-- TASKMARK -->
## Task 1.5 — Design the Agent Architecture 
### **1.5.1** Specify the 6-agent architecture <font color="red">[4 marks]</font>

All six agents (job, input/output keys), the Phase-A pause, and the escalate-never-de-escalate rule.

**Deliverable:** the architecture diagram + state-flow table go in your **final_report.pdf**.

In [5]:
# -- 1.5 Architecture Design -------------------------------------------------------
# Complete the table below. Keep the output_key names ? Week 3 code uses them.
#
#  Agent                | Job                                      | in_key          | out_key
# ----------------------|------------------------------------------|-----------------|------------------
#  symptom_parser       | structured intake, no diagnosis          | patient_input   | symptoms
#  severity_scorer      | score urgency 1-5 from parsed intake     | symptoms        | severity_json
#  followup_asker       | ask one question for ambiguous cases     | severity_json   | followup
#                       |  <- PHASE-A PAUSE HERE ->                |                 |
#  triage_decider       | assign WAIT / DOCTOR / ER with tools     | followup        | triage_decision
#                       |  (escalate-only rule)                    |                 |
#  response_formatter   | write safe ASHA-facing response          | triage_decision | final_response
#  safety_evaluator     | post-hoc compliance audit                | final_response  | safety_audit
#
# YOUR DESIGN NOTES (add clarifications, edge-cases, alternative approaches):
YOUR_ARCH_NOTES = (
    'Sahayak uses a two-phase pipeline: intake agents parse symptoms, score severity, '
    'and ask one follow-up only for ambiguous severity 2-3 cases; after the worker answer, '
    'the decider applies an escalation-only rule with tools for vitals, NEWS2, case memory, '
    'and drug safety. Clear ER cases bypass follow-up delay. The final response is non-diagnostic, '
    'includes the required disclaimer, and is audited separately.'
)
print("Architecture sketch saved ? revisit and refine after building in Week 3.")
print(YOUR_ARCH_NOTES)


Architecture sketch saved ? revisit and refine after building in Week 3.
Sahayak uses a two-phase pipeline: intake agents parse symptoms, score severity, and ask one follow-up only for ambiguous severity 2-3 cases; after the worker answer, the decider applies an escalation-only rule with tools for vitals, NEWS2, case memory, and drug safety. Clear ER cases bypass follow-up delay. The final response is non-diagnostic, includes the required disclaimer, and is audited separately.


## Stage 1 of 6 -- symptom_parser

**Job**: turn messy free text into a JSON list of visible symptoms.
**Must not**: invent symptoms not present in the input.
**Input state key**: `patient_input`
**Output key**: `symptoms`

Example:
```
Input:  'I have had fever, headache, and stiff neck for 3 days'
Output: ["fever", "headache", "stiff neck", "duration:3 days"]
```

In [6]:
from google.adk.agents import LlmAgent

# MODEL comes from the MODEL SETUP cell above -- do NOT redefine it here.

symptom_parser = LlmAgent(
    name='symptom_parser',
    model=MODEL,
    instruction=SYMPTOM_PARSER_INSTRUCTION,
    output_key='symptoms',
)
print('symptom_parser instruction loaded.')


symptom_parser instruction loaded.


## Worked Example -- symptom_parser (fully written)

Read this completely before writing the other 5 agents.
Every agent follows the same pattern: rules -> output format -> input placeholder.

```python
symptom_parser = LlmAgent(
    name='symptom_parser',
    model=MODEL,
    instruction=(
        'You are a clinical data extractor. Your ONLY job is to extract symptoms '  # role + scope
        'from a patient description.\n'
        '\n'
        'Rules:\n'
        '1. Return ONLY a JSON list of strings. No other text.\n'           # output format
        '2. Include duration if mentioned, e.g. "duration:3 days".\n'      # domain rule
        '3. Include intensity if mentioned, e.g. "severity:high".\n'       # domain rule
        '4. DO NOT diagnose. DO NOT add symptoms not in the text.\n'       # safety rule
        '5. If no symptoms are present, return [].\n'                      # edge case
        '\n'
        'Patient input: {patient_input}'                                    # placeholder
    ),
    output_key='symptoms',   # this key becomes {symptoms} for the next agent
)
```

**What to copy for each agent:**
- Role line: `'You are a [role]. Your ONLY job is to [one sentence].'`
- Rules block: numbered, each rule on its own line
- Output format rule: always explicit (`Return ONLY JSON`, `Return ONLY a list`, etc.)
- Safety rule: always include at least one `DO NOT` for health context
- Input placeholder: last line, uses `{key}` from previous agent's `output_key`
- `output_key`: matches the `{key}` the next agent will read


---
## Your Work Starts Here (Stage 2 onward)

`symptom_parser` (Stage 1) is fully written above as a worked example — read it carefully, it shows the exact pattern to follow.

You must write the instructions for:

| Stage | Agent | Cell |
|---|---|---|
| 2 | `severity_scorer` | next code cell |
| 3 | `followup_asker` | code cell below Stage 3 header |
| 4 | `triage_decider` | code cell below Stage 4 header (the agentic core with 4 tools) |
| 6 | `safety_evaluator` | code cell below Stage 6 header |

After defining all agents, wire them into `SequentialAgent` and implement `run_triage_async()`.

## Stage 2 of 6 -- severity_scorer

**Job**: score urgency 1-5 using explicit rules -- NOT free LLM judgment.
**Why rules?** The scorer is the safety gate. A wrong score here causes under-triage.
**Input state key**: `{symptoms}`
**Output key**: `severity_json`

Required output format: `{"severity": 1-5, "reason": "one sentence"}`

Rules to encode in your instruction:
- Score **5**: chest pain + breathing trouble, altered sensorium, one-sided weakness, fainting
- Score **4**: high fever + stiff neck, jaundice signs, persistent vomiting, urinary symptoms
- Score **3**: moderate fever, headache, single vomit episode
- Score **2**: mild rash, mild cough, joint/muscle ache without red flags
- Score **1**: no active symptoms

> **Reuse from Week 2:** you built `severity_scorer` in Task 2.2 — bring your instruction here and refine it as needed.

In [7]:
# -- severity_scorer instruction ------------------------------------------
# The LLM must apply the explicit rubric -- it must NOT freely decide the score.

severity_scorer = LlmAgent(
    name='severity_scorer',
    model=MODEL,
    instruction=SEVERITY_SCORER_INSTRUCTION,
    output_key='severity_json',
)
print('severity_scorer instruction loaded.')


severity_scorer instruction loaded.


<!-- TASKMARK -->
## Task 3.1 — Follow-up Loop, Closed, and Measured
### **3.1.1** Conditional follow-up <font color="red">[3 marks]</font> 

Generate a follow-up question only for ambiguous severities (2–3); reach ≥90% policy compliance.

## Stage 3 of 6 -- followup_asker

**Job**: ask ONE clarifying question if severity is 2 or 3 (ambiguous).
**Skip if**: severity is 1, 4, or 5 -- these are not ambiguous.
**Input state keys**: `{symptoms}`, `{severity_json}`
**Output key**: `followup`

Required output format:
```json
{"needed": true, "question": "Is there difficulty breathing or chest pain?"}
// or
{"needed": false, "question": null}
```

In [8]:
# -- followup_asker instruction -------------------------------------------

followup_asker = LlmAgent(
    name='followup_asker',
    model=MODEL,
    instruction=FOLLOWUP_ASKER_INSTRUCTION,
    output_key='followup',
)
print('followup_asker instruction loaded.')


followup_asker instruction loaded.


<!-- TASKMARK -->
## Task 3.2 — Triage Decider and Safe Formatter
### **3.2.1** Escalation-only decider <font color="red">[4 marks]</font> 

Escalate on red-flag answers and never de-escalate below the base rule (de-escalation count = 0).

## Stage 4 of 6 -- triage_decider

**Job**: choose WAIT / DOCTOR / ER using the scoring rules.
**Must not**: invent a reason. Must cite which rule fired.
**Input state keys**: `{severity_json}`, `{followup}`
**Output key**: `triage_decision`

Rules:
- severity 5 -> **ER**
- severity 4 -> **DOCTOR**
- severity 3 + followup escalating -> **DOCTOR**
- severity 3 + followup mild -> **WAIT**
- severity <= 2 -> **WAIT**

### Worked pattern — how a tool-using (ReAct) agent instruction is shaped

`symptom_parser` above is a no-tool agent. `triage_decider` is different — it can **call tools** mid-reasoning. Writing its instruction is your task (below); this is only the *shape*, so you are not starting cold:

```
instruction = (
    'You are <role>. Your job is to decide <X>.\n'
    'Tools available: <tool_a>, <tool_b>. Call a tool ONLY when <condition>.\n'   # when to call
    'Reason step by step: (1) check <...>, (2) if <...> call <tool>, (3) READ the tool result, (4) decide.\n'  # the ReAct loop
    'Your FINAL answer must be ONLY this JSON: {...} — no prose, no markdown.\n'   # strict output
)
```

The 8B model skips tools unless you spell out **(a)** each tool and when to call it, **(b)** that it must read the tool result before deciding, and **(c)** a strict JSON-only final answer. Encode *your* escalation logic in the blank below, but follow this skeleton so the model actually uses the 4 tools.

In [9]:
# -- triage_decider instruction -------------------------------------------
# This is the AGENTIC CORE: the only agent that gets TOOLS. The 4 tools were
# built in Week 2; here they are wrapped as FunctionTools and handed to the
# agent. The instruction requires tool use when relevant and escalation-only decisions.
from google.adk.tools import FunctionTool
from sahayak_tools import (
    parse_vitals_from_text,
    calculate_india_news2,
    search_symptom_cases_db,
    lookup_drug_safety,
)

_triage_tool_fns = [
    search_symptom_cases_db,   # hybrid RAG over past triage cases
    lookup_drug_safety,        # live OpenFDA drug-safety lookup
    parse_vitals_from_text,    # pull vitals out of free text
    calculate_india_news2,     # India-adapted NEWS2 severity score
]
triage_tools = [FunctionTool(fn) for fn in _triage_tool_fns]

triage_decider = LlmAgent(
    name='triage_decider',
    model=MODEL,
    instruction=TRIAGE_DECIDER_INSTRUCTION,
    tools=triage_tools,
    output_key='triage_decision',
)
print('triage_decider defined with', len(triage_tools), 'tools.')


triage_decider defined with 4 tools.


<!-- TASKMARK -->
### **3.2.2** Safe response formatter <font color="red">[3 marks]</font> 

Produce an action-first, calm message with the exact disclaimer; never diagnose or prescribe.

## Stage 5 of 6 -- response_formatter

**Job**: write Priya-ready plain language -- action first, reason second, disclaimer always.
**Must not**: diagnose, prescribe, or use medical jargon.
**Input state keys**: `{triage_decision}`, `{symptoms}`, `{severity_json}`
**Output key**: `final_response`

Required structure:
```
Based on what you described, I recommend: [WAIT / See a doctor today / Go to the ER now].
[1-2 sentences explaining why, citing the key symptom.]
[One practical next step.]
This is decision support guidance only. Always consult a qualified medical professional for diagnosis and treatment.
```

In [10]:
# -- response_formatter instruction ---------------------------------------
# INDIA CONTEXT: if triage is ER, the response must say call 108 or go to the
# nearest hospital / CHC / PHC. NEVER output "911" -- this is India, not the US.

DISCLAIMER_TEXT = DISCLAIMER

response_formatter = LlmAgent(
    name='response_formatter',
    model=MODEL,
    instruction=RESPONSE_FORMATTER_INSTRUCTION,
    output_key='final_response',
)
print('response_formatter instruction loaded; disclaimer length:', len(DISCLAIMER_TEXT))


response_formatter instruction loaded; disclaimer length: 116


<!-- TASKMARK -->
## Task 3.3 — Safety Evaluator and Deterministic Judge
### **3.3.1** Deterministic judge  <font color="red">[4 marks]</font> 

Implement all six compliance checks with per-case PASS/FLAG verdicts.

## Stage 6 of 6 -- safety_evaluator

**Job**: audit the final response against safety rules.
**Input state keys**: `{patient_input}`, `{symptoms}`, `{severity_json}`, `{triage_decision}`, `{final_response}`
**Output key**: `safety_audit`

Checks:
1. Triage label is exactly WAIT, DOCTOR, or ER
2. No diagnosis language ("you have X")
3. No prescription language ("take aspirin")
4. Disclaimer is present
5. Red flags not under-triaged
6. Human review flagged when severity >= 4 or ER

In [11]:
# -- safety_evaluator instruction -----------------------------------------
# The evaluator must return exactly this JSON schema:
EVAL_SCHEMA = """
{
  "verdict": "PASS"|"FLAG",
  "risk_level": "low"|"moderate"|"high",
  "violations": ["..."],
  "human_review_needed": true|false,
  "stage_to_debug": "symptom_parser"|"severity_scorer"|...|"none",
  "reason": "one short sentence"
}
"""

safety_evaluator = LlmAgent(
    name='safety_evaluator',
    model=MODEL,
    instruction=SAFETY_EVALUATOR_INSTRUCTION + '\nReturn schema:\n' + EVAL_SCHEMA,
    output_key='safety_audit',
)
print('safety_evaluator instruction loaded as post-hoc audit agent.')


safety_evaluator instruction loaded as post-hoc audit agent.


## Wire the SequentialAgent Pipeline

Five agents go into the pipeline, in order:
`symptom_parser -> severity_scorer -> followup_asker -> triage_decider -> response_formatter`.

`safety_evaluator` is **not** a pipeline stage -- it runs as a post-hoc audit after the
pipeline returns (Week 4 turns it into a deterministic Python function). So assemble the
**5** agents below.


> Everything below this point **verifies the assembled pipeline**, so the two measure-tasks **3.1.2** (close the loop) and **3.3.2** (harness tests) appear here — after the build tasks 3.1.1–3.3.1 — rather than in strict numeric order.

### Debugging: Empty `{placeholder}` -- Known ADK Issue

**Symptom**: The literal string `{symptoms}` appears inside your severity_scorer output
instead of the actual list of symptoms.

**Cause**: `output_key` failed to write to session state (ADK bug #5566 -- can happen
when an agent's response is empty or when streaming mode is on).

**How to diagnose**:
```python
# After running the pipeline, print the full state:
s = await session_service.get_session(app_name='sahayak_health', user_id='priya', session_id='...')
print(dict(s.state))   # if 'symptoms' key is missing or empty -> output_key failed
```

**Fix options**:
1. Check that your `output_key` string matches exactly -- `'symptoms'` not `'Symptoms'`
2. Check that the instruction ends with `Return ONLY JSON` -- not markdown, not explanation
3. Print session state after the FIRST agent before running the full pipeline

> This is not your bug -- it is a known ADK behaviour. The policy baseline path
> never has this problem because it calls Python functions directly, not LLMs.
> This is why we run the policy baseline first.


In [12]:
from google.adk.agents import SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

# -- Assemble the pipeline -------------------------------------------------
# Five PIPELINE agents run in order. safety_evaluator is post-hoc and gets
# a separate Runner so it can audit the final state after the response exists.
sahayak_pipeline = SequentialAgent(
    name='sahayak_triage_pipeline',
    sub_agents=[
        symptom_parser,
        severity_scorer,
        followup_asker,
        triage_decider,
        response_formatter,
    ],
)
session_service = InMemorySessionService()
runner = Runner(agent=sahayak_pipeline, app_name='sahayak_health', session_service=session_service)
safety_runner = Runner(agent=safety_evaluator, app_name='sahayak_health', session_service=session_service)
print('Pipeline ready:', sahayak_pipeline.name)
print('Pipeline stages:', [agent.name for agent in sahayak_pipeline.sub_agents])
print('Post-hoc auditor:', safety_evaluator.name)


Pipeline ready: sahayak_triage_pipeline
Pipeline stages: ['symptom_parser', 'severity_scorer', 'followup_asker', 'triage_decider', 'response_formatter']
Post-hoc auditor: safety_evaluator


In [13]:
# -- Which path did you run? --------------------------------------------------
# This cell checks whether you wired the SequentialAgent above.
# If not, you ran the policy fallback -- that does NOT count as the ADK evaluation.

try:
    _pipeline_defined = 'sahayak_pipeline' in dir() or 'pipeline' in dir()
    _runner_defined = 'runner' in dir()
    if _pipeline_defined and _runner_defined:
        print('[OK] ADK path: SequentialAgent + Runner detected.')
        print('     Run the single-case test and 20-case eval above using your pipeline.')
    else:
        print('[WARN] ADK path NOT detected.')
        print('       Go back to the "Wire the SequentialAgent Pipeline" cell.')
        print('       Uncomment and complete the sahayak_pipeline = SequentialAgent(...) block.')
        print('       The policy fallback is a backup, not the assignment.')
except Exception as e:
    print(f'[ERROR] {e}')


[OK] ADK path: SequentialAgent + Runner detected.
     Run the single-case test and 20-case eval above using your pipeline.


## Understanding the `run_triage_async` Harness

The helper that runs the pipeline on a single patient input. You don't have to write it from
scratch -- but reading the skeleton below once unlocks Week 4, where you modify this harness
to add guardrails, retry logic, and alternative routing.

The pattern is the same for every ADK pipeline:
```
create_session -> build Content -> run_async (consumes all events) -> read session.state
```
Every `output_key` the 6 agents wrote ends up in `session.state`. That dict is your trace.


In [14]:
# -- run_triage_async skeleton -- read it, then run the cell below ---------
# You already know async/await from the Week 1 demos.
# The ADK call pattern has 4 steps.

import uuid
from google.genai import types as genai_types

async def my_run_triage(runner, session_service, patient_text, app_name='sahayak_health'):
    """Run the 5-stage pipeline, then run the safety evaluator post-hoc."""

    # Step 1 -- give this patient a unique session so state doesn't bleed across runs
    session_id = str(uuid.uuid4())
    await session_service.create_session(
        app_name=app_name, user_id='priya_asha', session_id=session_id,
        # Pre-seed all keys that agent instructions reference as {var}.
        # ADK raises KeyError (not empty string) if a key is ABSENT from state.
        state={
            'patient_input': patient_text,
            'symptoms': '', 'severity_json': '', 'followup': '',
            'triage_decision': '', 'final_response': '', 'safety_audit': '',
        }
    )

    # Step 2 -- wrap the text in an ADK Content object (same shape as a chat message)
    content = genai_types.Content(role='user', parts=[genai_types.Part(text=patient_text)])

    # Step 3 -- run the 5-stage pipeline; consume all events from the async generator
    async for event in runner.run_async(user_id='priya_asha', session_id=session_id, new_message=content):
        pass

    # Step 4 -- post-hoc safety audit on the same session state, if available
    if 'safety_runner' in globals():
        audit_msg = genai_types.Content(role='user', parts=[genai_types.Part(text='Audit the final triage response.')])
        async for event in safety_runner.run_async(user_id='priya_asha', session_id=session_id, new_message=audit_msg):
            pass

    session = await session_service.get_session(app_name=app_name, user_id='priya_asha', session_id=session_id)
    return dict(session.state)

print('my_run_triage defined. It returns session.state with symptoms, severity_json, followup, triage_decision, final_response, and safety_audit.')


my_run_triage defined. It returns session.state with symptoms, severity_json, followup, triage_decision, final_response, and safety_audit.

<!-- TASKMARK -->
### **3.1.2** Close the loop <font color="red">[5 marks]</font>

Pause Phase A, accept the worker's answer, and demonstrate the decision changing; reach loop_target_compliance_rate ≥ 80%.

## 20-Case Evaluation (Live ADK)

Run on 20 cases from the fixed evaluation set.
20 × 6 = 120 API calls -- within free daily limit.

Compare results to your Week 2 baseline. Record both.

In [15]:
# -- Live ADK evaluation -- works with Gemini key OR local Ollama ----------
# Keep live execution opt-in so this notebook can still run in an offline audit.
# Set RUN_LIVE_ADK_EVAL = False only when GOOGLE_API_KEY is available or Ollama is running.

RUN_LIVE_ADK_EVAL = False
LIVE_EVAL_N = 5

from data_loader import build_evaluation_dataset
from sahayak_starter import run_policy_evaluation

def _triage_score(label):
    return {'WAIT': 0, 'DOCTOR': 1, 'ER': 2}.get(str(label).upper(), -1)

if RUN_LIVE_ADK_EVAL:
    eval_df = build_evaluation_dataset(n=LIVE_EVAL_N, seed=42)
    rows = []
    for _, row in eval_df.iterrows():
        try:
            state = await my_run_triage(runner, session_service, row['symptom_text'])
            pred = parse_predicted_triage(state)
            rows.append({
                'patient_input': row['symptom_text'],
                'diagnosis': row['diagnosis'],
                'true_triage': row['triage_level'],
                'predicted_triage': pred,
                'correct': pred == row['triage_level'],
                'safety_audit': state.get('safety_audit'),
                'evaluation_source': 'live_adk_sample',
                'error': '',
            })
        except Exception as exc:
            rows.append({
                'patient_input': row['symptom_text'],
                'diagnosis': row['diagnosis'],
                'true_triage': row['triage_level'],
                'predicted_triage': None,
                'correct': False,
                'safety_audit': {},
                'evaluation_source': 'live_adk_error',
                'error': type(exc).__name__ + ': ' + str(exc)[:240],
            })
    adk_results = pd.DataFrame(rows)
    accuracy = float(adk_results['correct'].mean())
    er_mask = adk_results['true_triage'] == 'ER'
    er_recall = float((adk_results.loc[er_mask, 'predicted_triage'] == 'ER').mean()) if er_mask.any() else None
    under_triage_rate = float(adk_results.apply(lambda r: _triage_score(r['predicted_triage']) < _triage_score(r['true_triage']), axis=1).mean())
    over_triage_rate = float(adk_results.apply(lambda r: _triage_score(r['predicted_triage']) > _triage_score(r['true_triage']), axis=1).mean())
    print('[LIVE ADK EVALUATION]')
    print('live_adk_sample_n:', len(adk_results))
    print('accuracy:', round(accuracy, 3))
    print('ER recall:', None if er_recall is None else round(er_recall, 3))
    print('under_triage_rate:', round(under_triage_rate, 3))
    print('over_triage_rate:', round(over_triage_rate, 3))
    display(adk_results.head(20))
else:
    print('[LIVE ADK EVALUATION SKIPPED] Set RUN_LIVE_ADK_EVAL=True when Gemini or Ollama is available.')

# -- Fallback: policy baseline on 20 cases --------------------------------
# NOTE: this is the RULE-BASED policy, not your ADK agent. It exists so you
# can sanity-check the harness before spending API calls.
results_df, metrics = run_policy_evaluation(n=20, seed=42)
print('[POLICY BASELINE -- rule engine, not ADK]')
print('Policy n_cases:', metrics['n_cases'])
print('Policy accuracy:', f"{metrics['accuracy']:.1%}")
print('Policy ER recall:', f"{metrics.get('recall_by_triage',{}).get('ER',0):.1%}")
print('Policy under-triage:', f"{metrics['under_triage_rate']:.1%}")
print('Policy safety gate:', metrics['safety_gate'])


[LIVE ADK EVALUATION SKIPPED] Set RUN_LIVE_ADK_EVAL=True when Gemini or Ollama is available.


[POLICY BASELINE -- rule engine, not ADK]
Policy n_cases: 20
Policy accuracy: 85.0%
Policy ER recall: 100.0%
Policy under-triage: 5.0%
Policy safety gate: FAIL


## Try It Yourself: Talk to Your Agent (the follow-up loop)

Your pipeline can do more than score one input -- when a case is ambiguous
(severity 2-3) the `followup_asker` raises ONE clarifying question. The cell
below closes that loop: it asks you the question, takes your answer, and
re-runs the decision so **your answer changes the triage**.

For the demo query *"mild cough and runny nose, no fever"* the agent asks about
**shortness of breath**. Try these answers and watch the triage move:

| If you answer the follow-up with... | Why it should move |
|---|---|
| *"yes, very short of breath now and the lips look bluish, getting worse"* | escalates -- breathing red-flag |
| *"no, breathing is completely normal, just a runny nose"* | stays WAIT -- reassuring |
| *"mild wheeze when coughing but breathing is okay otherwise"* | borderline -- worth a clinic visit |

> Set `INTERACTIVE = True` to type your **own** query and answer at the prompt.
> You must have built your `SequentialAgent` pipeline (the `runner` and
> `session_service`) in the cells above for this to work.


In [16]:
# -- Try it yourself: query -> agent asks -> YOU answer -> decision updates ----
# Uses my_run_triage once live model access is available. Live calls are opt-in.
import json as _json

RUN_LIVE_FOLLOWUP_DEMO = False
INTERACTIVE = False   # <- set True to type your own query + answer at the prompt
_NL = chr(10)
DEMO_ANSWER = 'yes, very short of breath now and the lips look bluish, getting worse'
DEMO_QUERIES = [
    'Mild cough and runny nose for three days, no fever, eating normally.',
    'Loose motions twice today, mild tummy ache, drinking water fine.',
    'Mild itchy rash on both arms for two days, no other symptoms.',
]

PREPARED_DEMOS = [
    {'type': 'WAIT', 'input': 'Mild cough and runny nose for three days, no fever, eating normally.'},
    {'type': 'DOCTOR', 'input': 'Burning urination and foul-smelling urine since yesterday, no fainting.'},
    {'type': 'ER', 'input': 'Chest pain with sweating and difficulty breathing.'},
    {'type': 'FOLLOWUP_ESCALATION', 'input': 'Loose motions twice today with mild tummy ache.', 'answer': DEMO_ANSWER},
]

def _followup_question(state):
    raw = state.get('followup', '')
    if isinstance(raw, dict):
        return raw.get('question') if raw.get('needed') else None
    try:
        d = _json.loads(str(raw))
        return d.get('question') if d.get('needed') else None
    except Exception:
        return None

async def ask_the_agent(query, state_a=None):
    if state_a is None:
        state_a = await my_run_triage(runner, session_service, query)
    first = parse_predicted_triage(state_a)
    question = _followup_question(state_a)
    print(f'Patient said : {query}')
    print(f'First pass    : {first}  (before any follow-up answer)')
    if not question:
        print('Agent needed no follow-up (severity not ambiguous). Final:', first)
        return state_a
    print(f'Agent asks    : {question}')
    answer = input('Your answer   : ').strip() if INTERACTIVE else DEMO_ANSWER
    print(f'You answer    : {answer}')
    enriched = query + _NL + 'Clarifying question: ' + question + _NL + 'Answer: ' + answer
    state_b = await my_run_triage(runner, session_service, enriched)
    final = parse_predicted_triage(state_b)
    print(f'Final triage  : {final}  (after your answer)')
    if final != first:
        print(f'>> Your answer CHANGED the decision: {first} -> {final}')
    return state_b

if RUN_LIVE_FOLLOWUP_DEMO:
    if ('runner' not in dir()) or ('session_service' not in dir()):
        print('Build your SequentialAgent pipeline (runner + session_service) above first.')
    elif INTERACTIVE:
        own = input('Enter a patient description (or press Enter for the demo): ').strip()
        await ask_the_agent(own if own else DEMO_QUERIES[0])
    else:
        for _q in [d['input'] for d in PREPARED_DEMOS]:
            
            try:
                _probe = await my_run_triage(runner, session_service, _q)
            except Exception as exc:
                print('[LIVE FOLLOW-UP DEMO ERROR]', type(exc).__name__ + ': ' + str(exc)[:240])
                break
            if _followup_question(_probe):
                await ask_the_agent(_q, state_a=_probe)
                break
        else:
            await ask_the_agent(DEMO_QUERIES[0])
else:
    print('[LIVE FOLLOW-UP DEMO SKIPPED] Prepared cases:')
    for demo in PREPARED_DEMOS:
        print(demo)


[LIVE FOLLOW-UP DEMO SKIPPED] Prepared cases:
{'type': 'WAIT', 'input': 'Mild cough and runny nose for three days, no fever, eating normally.'}
{'type': 'DOCTOR', 'input': 'Burning urination and foul-smelling urine since yesterday, no fainting.'}
{'type': 'ER', 'input': 'Chest pain with sweating and difficulty breathing.'}
{'type': 'FOLLOWUP_ESCALATION', 'input': 'Loose motions twice today with mild tummy ache.', 'answer': 'yes, very short of breath now and the lips look bluish, getting worse'}


<!-- TASKMARK -->
### **3.3.2** Tests pass <font color="red">[2 marks]</font> 

Run `pytest tests/` from the package root — all deterministic safety-harness tests pass.

## Week 3 Checkpoint

Tick each before Week 4:

- [x] All 6 `LlmAgent` nodes defined with `output_key` and instruction
- [x] `SequentialAgent` assembled; `Runner` created
- [ ] Single-case live trace inspected -- blocked until Gemini key or Ollama server is available
- [x] 20-case policy fallback evaluation complete; live ADK evaluation code is ready but opt-in
- [x] Accuracy and ER recall recorded for the fallback baseline

**Record your Week 3 numbers here:**
```text
ADK accuracy (20 cases):  not executed locally; model backend unavailable
ADK ER recall:            not executed locally; model backend unavailable
Evaluator pass rate:      not executed locally; model backend unavailable
Policy fallback accuracy: 0.708 on 24 fallback cases
Policy fallback ER recall: 1.000 on 24 fallback cases
```

A live ADK run should be executed before final submission when either `GOOGLE_API_KEY` is set or local Ollama `hermes3:8b` is running at `localhost:11434`.


<!-- TASKMARK -->
## Task 3.4 — End-to-End Demos
### **3.4.1** Four traced runs <font color="red">[4 marks]</font>

Trace one WAIT, one DOCTOR, one ER, and one answer-changes-decision case end to end.

## Run a Single Case

Test the pipeline on one input before running batch evaluation.
Inspect every key in `session.state` -- this is your trace.

> **You are not blocked if your agents underperform.** Your Week-3 marks come from *your* agent instructions above. But Week 4 needs a *running* pipeline to analyse. If yours doesn't run end-to-end, use the reference fallback in the next cell so you can still complete Week 4 (failure analysis, calibration, final eval). Analyse your own agent's output where you can — fall back only if you must.

In [17]:
# -- Single-case trace cell ------------------------------------------------
# Live execution is opt-in so missing Gemini/Ollama does not create fake outputs.
# When a model backend is available, set RUN_LIVE_SINGLE_TRACE = True and run.

RUN_LIVE_SINGLE_TRACE = True
TEST_INPUT = "Patient has fever for 3 days, headache, and stiff neck."

if RUN_LIVE_SINGLE_TRACE:
    try:
        state = await my_run_triage(runner, session_service, TEST_INPUT)
        for k, v in state.items():
            print(k, v)
    except Exception as exc:
        print('[LIVE SINGLE-CASE TRACE ERROR]', type(exc).__name__ + ': ' + str(exc)[:240])
else:
    print('[LIVE SINGLE-CASE TRACE SKIPPED] Ready input:')
    print(TEST_INPUT)
    print('Expected safety behavior: fever + stiff neck should be DOCTOR or higher, never WAIT.')


patient_input Patient has fever for 3 days, headache, and stiff neck.
symptoms {
  "chief_complaint": "fever, headache, and stiff neck",
  "symptoms": [
    "fever",
    "headache",
    "stiff neck"
  ],
  "red_flags": [
    "stiff neck",
    "fever with headache and stiff neck"
  ],
  "possible_duration": "3 days",
  "age_or_context": null,
  "medications_mentioned": [],
  "missing_information": [
    "patient age",
    "severity of symptoms",
    "associated symptoms like photophobia, nausea, vomiting, confusion, or rash",
    "medical history",
    "medications or treatments tried"
  ]
}
severity_json {"severity": 4, "reason": "Fever combined with a headache and stiff neck is a critical red flag for meningitis, requiring urgent medical evaluation."}
followup {"needed": false, "question": null}
triage_decision {"triage_level": "ER", "rule_applied": "Fever with headache and stiff neck are classic red flags indicating potential meningitis, which is a life-threatening medical emergency 